In [ ]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("HDB_Resale_Analysis") \
    .getOrCreate()

path_to_data = "D:/TAI_LIEU/HK6/BIGDATA/FINAL_PROJECT/big-data/data_source/" 

df_resale = spark.read.csv(path_to_data + "resale.csv", header=True, inferSchema=True)
df_flat = spark.read.csv(path_to_data + "flat.csv", header=True, inferSchema=True)
df_lease = spark.read.csv(path_to_data + "lease.csv", header=True, inferSchema=True)
df_location = spark.read.csv(path_to_data + "location.csv", header=True, inferSchema=True)
df_time = spark.read.csv(path_to_data + "time.csv", header=True, inferSchema=True)

# Tạo Temp View để thực thi Spark SQL
df_resale.createOrReplaceTempView("resale")
df_flat.createOrReplaceTempView("flat")
df_lease.createOrReplaceTempView("lease")
df_location.createOrReplaceTempView("location")
df_time.createOrReplaceTempView("time")

print("Đã load xong dữ liệu và sẵn sàng query!")

Đã load xong dữ liệu và sẵn sàng query!


In [ ]:

dataframes = {
    "RESALE": df_resale,
    "FLAT": df_flat,
    "LEASE": df_lease,
    "LOCATION": df_location,
    "TIME": df_time
}


print("\n" + "="*50)
print("KIỂM TRA CẤU TRÚC DỮ LIỆU (SCHEMA)")
print("="*50)

for table_name, df in dataframes.items():
    print(f"\n---> BẢNG: {table_name}")
    df.printSchema()


KIỂM TRA CẤU TRÚC DỮ LIỆU (SCHEMA)

---> BẢNG: RESALE
root
 |-- transaction_id: integer (nullable = true)
 |-- time_id: integer (nullable = true)
 |-- location_id: integer (nullable = true)
 |-- flat_id: integer (nullable = true)
 |-- lease_id: integer (nullable = true)
 |-- resale_price: double (nullable = true)


---> BẢNG: FLAT
root
 |-- flat_id: integer (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)


---> BẢNG: LEASE
root
 |-- lease_id: integer (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- remaining_lease: string (nullable = true)


---> BẢNG: LOCATION
root
 |-- town: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- block: string (nullable = true)
 |-- location_id: integer (nullable = true)


---> BẢNG: TIME
root
 |-- time_id: integer (nullable = true)
 |-- month: timestamp (nullable = 

In [7]:
# ==============================================================================
# CÂU 1 (VĨ MÔ): XU HƯỚNG THANH KHOẢN VÀ TỐC ĐỘ TĂNG TRƯỞNG (MoM GROWTH)
# Insight: Xác định các giai đoạn bùng nổ hoặc đóng băng của thị trường.
# ==============================================================================
query_1 = """
WITH MonthlyMetrics AS (
    SELECT 
        t.year, 
        t.month_num, 
        COUNT(r.transaction_id) AS total_transactions,
        SUM(r.resale_price) AS total_revenue
    FROM resale r
    JOIN time t ON r.time_id = t.time_id
    GROUP BY t.year, t.month_num
),
Trend AS (
    SELECT 
        year, month_num, total_transactions, total_revenue,
        LAG(total_revenue) OVER(ORDER BY year, month_num) AS prev_revenue
    FROM MonthlyMetrics
)
SELECT 
    year, month_num, total_transactions, total_revenue, prev_revenue,
    ROUND(((total_revenue - prev_revenue) / prev_revenue) * 100, 2) AS revenue_growth_pct
FROM Trend
WHERE prev_revenue IS NOT NULL
ORDER BY year, month_num
"""
print("\n--- [1] XU HƯỚNG TĂNG TRƯỞNG DOANH THU THEO THÁNG ---")
spark.sql(query_1).show(10)




--- [1] XU HƯỚNG TĂNG TRƯỞNG DOANH THU THEO THÁNG ---
+----+---------+------------------+--------------+--------------+------------------+
|year|month_num|total_transactions| total_revenue|  prev_revenue|revenue_growth_pct|
+----+---------+------------------+--------------+--------------+------------------+
|2017|        2|              1080|  4.83079627E8|  5.02596777E8|             -3.88|
|2017|        3|              1889|8.4032589388E8|  4.83079627E8|             73.95|
|2017|        4|              1821|7.9860523164E8|8.4032589388E8|             -4.96|
|2017|        5|              1961|   8.7002365E8|7.9860523164E8|              8.94|
|2017|        6|              1733|  7.77027057E8|   8.7002365E8|            -10.69|
|2017|        7|              1765|  7.73923654E8|  7.77027057E8|              -0.4|
|2017|        8|              1941|  8.63540752E8|  7.73923654E8|             11.58|
|2017|        9|              1665|  7.53680876E8|  8.63540752E8|            -12.72|
|2017|    

In [8]:
# ==============================================================================
# CÂU 2 (VI MÔ): MỨC ĐỘ TẬP TRUNG DÒNG TIỀN (MARKET SHARE & PARETO)
# Insight: Xác định Top các "Town" đang gánh vác doanh thu của toàn quốc gia.
# ==============================================================================
query_2 = """
WITH TownSales AS (
    SELECT 
        loc.town, 
        SUM(r.resale_price) AS town_revenue
    FROM resale r
    JOIN location loc ON r.location_id = loc.location_id
    GROUP BY loc.town
),
RankedTowns AS (
    SELECT 
        town, 
        town_revenue,
        SUM(town_revenue) OVER() AS grand_total,
        DENSE_RANK() OVER(ORDER BY town_revenue DESC) AS rank
    FROM TownSales
)
SELECT 
    rank, town, town_revenue, grand_total,
    ROUND((town_revenue / grand_total) * 100, 2) AS market_share_pct
FROM RankedTowns
ORDER BY rank
"""
print("\n--- [2] THỊ PHẦN DOANH THU THEO TỪNG KHU VỰC ---")
spark.sql(query_2).show(10)


--- [2] THỊ PHẦN DOANH THU THEO TỪNG KHU VỰC ---
+----+-------------+--------------------+-----------------+----------------+
|rank|         town|        town_revenue|      grand_total|market_share_pct|
+----+-------------+--------------------+-----------------+----------------+
|   1|     SENGKANG|     7.44332371264E9|9.011005512181E10|            8.26|
|   2|      PUNGGOL|     6.92709548676E9|9.011005512181E10|            7.69|
|   3|     TAMPINES|       6.381075729E9|9.011005512181E10|            7.08|
|   4|    WOODLANDS|     5.76596138176E9|9.011005512181E10|             6.4|
|   5|       YISHUN|     5.28299855676E9|9.011005512181E10|            5.86|
|   6|  JURONG WEST|     5.27451134251E9|9.011005512181E10|            5.85|
|   7|      HOUGANG|     4.50666593952E9|9.011005512181E10|             5.0|
|   8|        BEDOK|       4.397340935E9|9.011005512181E10|            4.88|
|   9|  BUKIT MERAH|4.2547899136400003E9|9.011005512181E10|            4.72|
|  10|CHOA CHU KANG|     3

In [9]:
# ==============================================================================
# CÂU 3 (YẾU TỐ VẬT LÝ): ĐỊNH LƯỢNG GIÁ TRỊ TẦM NHÌN (STOREY PREMIUM)
# Insight: Chứng minh tầng càng cao giá mét vuông càng đắt thông qua kỹ thuật Pivot.
# ==============================================================================
query_3 = """
SELECT 
    loc.town,
    ROUND(AVG(CASE WHEN f.storey_range IN ('01 TO 03', '04 TO 06') THEN r.resale_price / f.floor_area_sqm END), 2) AS low_floor_sqm_price,
    ROUND(AVG(CASE WHEN f.storey_range IN ('07 TO 09', '10 TO 12', '13 TO 15') THEN r.resale_price / f.floor_area_sqm END), 2) AS mid_floor_sqm_price,
    ROUND(AVG(CASE WHEN f.storey_range NOT IN ('01 TO 03', '04 TO 06', '07 TO 09', '10 TO 12', '13 TO 15') THEN r.resale_price / f.floor_area_sqm END), 2) AS high_floor_sqm_price
FROM resale r
JOIN location loc ON r.location_id = loc.location_id
JOIN flat f ON r.flat_id = f.flat_id
GROUP BY loc.town
ORDER BY loc.town
"""
print("\n--- [3] SO SÁNH GIÁ MÉT VUÔNG THEO PHÂN KHÚC TẦNG ---")
spark.sql(query_3).show(10)



--- [3] SO SÁNH GIÁ MÉT VUÔNG THEO PHÂN KHÚC TẦNG ---
+-------------+-------------------+-------------------+--------------------+
|         town|low_floor_sqm_price|mid_floor_sqm_price|high_floor_sqm_price|
+-------------+-------------------+-------------------+--------------------+
|   ANG MO KIO|            5805.99|             9918.5|              7443.0|
|        BEDOK|            5853.48|            7493.76|             6902.47|
|       BISHAN|             8268.6|            9418.05|                NULL|
|  BUKIT BATOK|            5723.43|            6781.38|             4596.77|
|  BUKIT MERAH|             7316.9|             9786.6|             6658.19|
|BUKIT PANJANG|            5504.31|            6639.52|             4573.11|
|  BUKIT TIMAH|            8915.89|           10348.42|                NULL|
| CENTRAL AREA|            6121.05|            9013.19|             12225.5|
|CHOA CHU KANG|            5371.88|            6873.37|             4104.71|
|     CLEMENTI|      

In [10]:
# ==============================================================================
# CÂU 4 (YẾU TỐ PHÁP LÝ): KHẤU HAO TÀI SẢN THEO TUỔI THỌ HỢP ĐỒNG
# Insight: Cắt chuỗi xử lý dữ liệu Lease, phân nhóm xem giá trị rớt như thế nào khi nhà cũ đi.
# ==============================================================================
query_4 = """
WITH ParsedLease AS (
    SELECT 
        r.resale_price, 
        f.floor_area_sqm,
        CAST(SPLIT(l.remaining_lease, ' ')[0] AS INT) AS remaining_years
    FROM resale r
    JOIN lease l ON r.lease_id = l.lease_id
    JOIN flat f ON r.flat_id = f.flat_id
)
SELECT 
    CASE 
        WHEN remaining_years >= 90 THEN '1. Rat moi (>= 90 nam)'
        WHEN remaining_years >= 70 THEN '2. Trung binh (70 - 89 nam)'
        WHEN remaining_years >= 50 THEN '3. Cu (50 - 69 nam)'
        ELSE '4. Rat cu (< 50 nam)'
    END AS lease_status,
    COUNT(*) AS total_sales,
    ROUND(AVG(resale_price / floor_area_sqm), 2) AS avg_price_per_sqm
FROM ParsedLease
GROUP BY 
    CASE 
        WHEN remaining_years >= 90 THEN '1. Rat moi (>= 90 nam)'
        WHEN remaining_years >= 70 THEN '2. Trung binh (70 - 89 nam)'
        WHEN remaining_years >= 50 THEN '3. Cu (50 - 69 nam)'
        ELSE '4. Rat cu (< 50 nam)'
    END
ORDER BY lease_status
"""
print("\n--- [4] ĐÁNH GIÁ MỨC ĐỘ KHẤU HAO THEO THỜI HẠN THUÊ ---")
spark.sql(query_4).show()



--- [4] ĐÁNH GIÁ MỨC ĐỘ KHẤU HAO THEO THỜI HẠN THUÊ ---
+--------------------+-----------+-----------------+
|        lease_status|total_sales|avg_price_per_sqm|
+--------------------+-----------+-----------------+
|1. Rat moi (>= 90...|      42022|          7016.51|
|2. Trung binh (70...|      62030|          6996.23|
| 3. Cu (50 - 69 nam)|      72725|           5947.0|
|4. Rat cu (< 50 nam)|       4197|          3939.07|
+--------------------+-----------+-----------------+



In [11]:
# ==============================================================================
# CÂU 5 (KIỂM SOÁT RỦI RO): PHÁT HIỆN GIAO DỊCH BẤT THƯỜNG (Z-SCORE ANOMALY)
# Insight: Tìm ra những giao dịch bị đẩy giá lên cao bất hợp lý (> 2.5 độ lệch chuẩn).
# ==============================================================================
query_5 = """
WITH TownStats AS (
    SELECT 
        loc.town, 
        f.flat_type,
        AVG(r.resale_price) AS mean_price,
        STDDEV(r.resale_price) AS std_price
    FROM resale r
    JOIN location loc ON r.location_id = loc.location_id
    JOIN flat f ON r.flat_id = f.flat_id
    GROUP BY loc.town, f.flat_type
)
SELECT 
    r.transaction_id, 
    loc.town, 
    f.flat_type, 
    r.resale_price,
    ROUND(s.mean_price, 2) AS mean_price,
    ROUND(s.std_price, 2) AS std_price,
    ROUND((r.resale_price - s.mean_price) / s.std_price, 2) AS z_score
FROM resale r
JOIN location loc ON r.location_id = loc.location_id
JOIN flat f ON r.flat_id = f.flat_id
JOIN TownStats s ON loc.town = s.town AND f.flat_type = s.flat_type
WHERE r.resale_price > (s.mean_price + 2.5 * s.std_price) 
ORDER BY z_score DESC
"""
print("\n--- [5] DANH SÁCH GIAO DỊCH ĐỘT BIẾN GIÁ (OUTLIERS) ---")
spark.sql(query_5).show(10)


--- [5] DANH SÁCH GIAO DỊCH ĐỘT BIẾN GIÁ (OUTLIERS) ---
+--------------+-----------+---------+------------+----------+---------+-------+
|transaction_id|       town|flat_type|resale_price|mean_price|std_price|z_score|
+--------------+-----------+---------+------------+----------+---------+-------+
|        176059|BUKIT BATOK|   5 ROOM|    915000.0| 345007.53| 66040.58|   8.63|
|        144655|    PUNGGOL|   3 ROOM|   1220000.0| 491337.67| 95323.77|   7.64|
|        131956|    PUNGGOL|   3 ROOM|   1198000.0| 491337.67| 95323.77|   7.41|
|        161751|    PUNGGOL|   3 ROOM|   1150000.0| 491337.67| 95323.77|   6.91|
|        175664|     YISHUN|   3 ROOM|   1200000.0| 420632.61| 121128.4|   6.43|
|        170372|    PUNGGOL|   3 ROOM|   1100888.0| 491337.67| 95323.77|   6.39|
|        154543|BUKIT MERAH|   2 ROOM|   1500000.0| 293344.12|200025.97|   6.03|
|        145680|     YISHUN|   3 ROOM|   1080000.0| 420632.61| 121128.4|   5.44|
|        178035|     YISHUN|   3 ROOM|   1080000.0| 